# Experiment 9 — Universal Grammar Structural Probes

Central question: **does the universal concept space derived from multilingual embeddings
exhibit UG-like structural properties without any explicit grammatical supervision?**

If yes: UG properties are *informational attractors*, not biological constraints — they emerge
from communication pressure alone. The universal language inherits them for free.
If no: we need to explicitly inject UG structure — the design task from the PDF.

### Five probes:
1. **Category emergence** — does k-means on centroids recover noun/verb/emotion POS without labels?
2. **Argument structure** — are transitive vs intransitive verb centroids linearly separable?
3. **Economy / Zipf** — do pairwise centroid distances follow a power law?
4. **Cross-linguistic RSA** — is D_universal the best common structure across all D_lang?
5. **Semantic directions** — is there a consistent positive/negative axis in the concept space?

In [ ]:
!pip install sentence-transformers matplotlib scipy scikit-learn -q

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sentence_transformers import SentenceTransformer
from scipy.spatial.distance import pdist, squareform, cosine as cos_dist
from scipy.stats import spearmanr, pearsonr
from sklearn.cluster import KMeans
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import accuracy_score
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings('ignore')

try:
    import umap
    HAS_UMAP = True
except ImportError:
    HAS_UMAP = False

model = SentenceTransformer('LaBSE')
print('LaBSE loaded')

In [ ]:
# ── Extended concept vocabulary with POS and category labels ─────────────────
# We keep labels ONLY for evaluation — they are never used in computing centroids
# (the probes are unsupervised)

CONCEPT_DEFS = [
    # (concept, POS_category, transitivity, valence)
    # POS: 'concrete_noun', 'abstract_noun', 'emotion', 'transitive_verb', 'intransitive_verb'
    # valence: 'positive', 'negative', 'neutral'
    ('water',    'concrete_noun',     None,          'neutral'),
    ('fire',     'concrete_noun',     None,          'neutral'),
    ('earth',    'concrete_noun',     None,          'neutral'),
    ('sky',      'concrete_noun',     None,          'neutral'),
    ('wind',     'concrete_noun',     None,          'neutral'),
    ('stone',    'concrete_noun',     None,          'neutral'),
    ('river',    'concrete_noun',     None,          'neutral'),
    ('mountain', 'concrete_noun',     None,          'neutral'),
    ('forest',   'concrete_noun',     None,          'neutral'),
    ('child',    'concrete_noun',     None,          'neutral'),
    ('elder',    'concrete_noun',     None,          'neutral'),
    ('friend',   'concrete_noun',     None,          'positive'),
    ('enemy',    'concrete_noun',     None,          'negative'),
    ('time',     'abstract_noun',     None,          'neutral'),
    ('life',     'abstract_noun',     None,          'positive'),
    ('death',    'abstract_noun',     None,          'negative'),
    ('peace',    'abstract_noun',     None,          'positive'),
    ('war',      'abstract_noun',     None,          'negative'),
    ('hope',     'abstract_noun',     None,          'positive'),
    ('dream',    'abstract_noun',     None,          'positive'),
    ('change',   'abstract_noun',     None,          'neutral'),
    ('love',     'emotion',           None,          'positive'),
    ('fear',     'emotion',           None,          'negative'),
    ('trust',    'emotion',           None,          'positive'),
    ('joy',      'emotion',           None,          'positive'),
    ('pain',     'emotion',           None,          'negative'),
    ('give',     'transitive_verb',   'transitive',  'positive'),
    ('take',     'transitive_verb',   'transitive',  'neutral'),
    ('speak',    'transitive_verb',   'transitive',  'neutral'),
    ('think',    'transitive_verb',   'transitive',  'neutral'),
    ('find',     'transitive_verb',   'transitive',  'positive'),
    ('lose',     'transitive_verb',   'transitive',  'negative'),
    ('build',    'transitive_verb',   'transitive',  'positive'),
    ('break',    'transitive_verb',   'transitive',  'negative'),
    ('eat',      'transitive_verb',   'transitive',  'neutral'),
    ('feel',     'transitive_verb',   'transitive',  'neutral'),
    ('sleep',    'intransitive_verb', 'intransitive', 'neutral'),
    ('run',      'intransitive_verb', 'intransitive', 'neutral'),
    ('light',    'concrete_noun',     None,          'positive'),
    ('dark',     'abstract_noun',     None,          'negative'),
]

CONCEPTS   = [d[0] for d in CONCEPT_DEFS]
POS_LABELS = [d[1] for d in CONCEPT_DEFS]
TRANS      = [d[2] for d in CONCEPT_DEFS]
VALENCE    = [d[3] for d in CONCEPT_DEFS]

print(f'Concept pool: {len(CONCEPTS)} concepts')
from collections import Counter
print('POS distribution:', dict(Counter(POS_LABELS)))

In [ ]:
# ── Multilingual vocabulary + speaker-weighted centroids ─────────────────────
# Same 10 languages as Exp 7 v2
SPEAKER_WEIGHTS = {
    'en':1500, 'zh':1100, 'hi':600, 'es':560,  'ar':380,
    'ru':260,  'pt':260,  'fr':280, 'de':130,  'ja':125,
}

LANG_VOCAB = {
    'en': ['water','fire','earth','sky','wind','stone','river','mountain','forest','child',
           'elder','friend','enemy','time','life','death','peace','war','hope','dream',
           'change','love','fear','trust','joy','pain','give','take','speak','think',
           'find','lose','build','break','eat','feel','sleep','run','light','dark'],
    'zh': ['水','火','土','天空','风','石头','河流','山','森林','孩子',
           '老人','朋友','敌人','时间','生命','死亡','和平','战争','希望','梦想',
           '改变','爱','恐惧','信任','喜悦','痛苦','给','拿','说话','思考',
           '找到','失去','建造','破坏','吃','感觉','睡觉','跑','光','暗'],
    'hi': ['पानी','आग','पृथ्वी','आकाश','हवा','पत्थर','नदी','पहाड़','जंगल','बच्चा',
           'बुजुर्ग','दोस्त','दुश्मन','समय','जीवन','मृत्यु','शांति','युद्ध','आशा','सपना',
           'बदलाव','प्यार','डर','विश्वास','खुशी','दर्द','देना','लेना','बोलना','सोचना',
           'ढूंढना','खोना','बनाना','तोड़ना','खाना','महसूस करना','सोना','दौड़ना','रोशनी','अंधेरा'],
    'es': ['agua','fuego','tierra','cielo','viento','piedra','río','montaña','bosque','niño',
           'anciano','amigo','enemigo','tiempo','vida','muerte','paz','guerra','esperanza','sueño',
           'cambio','amor','miedo','confianza','alegría','dolor','dar','tomar','hablar','pensar',
           'encontrar','perder','construir','romper','comer','sentir','dormir','correr','luz','oscuridad'],
    'ar': ['ماء','نار','أرض','سماء','ريح','حجر','نهر','جبل','غابة','طفل',
           'مسن','صديق','عدو','وقت','حياة','موت','سلام','حرب','أمل','حلم',
           'تغير','حب','خوف','ثقة','فرح','ألم','أعطى','أخذ','تكلم','فكر',
           'وجد','فقد','بنى','كسر','أكل','شعر','نوم','ركض','ضوء','ظلام'],
    'ru': ['вода','огонь','земля','небо','ветер','камень','река','гора','лес','ребёнок',
           'старик','друг','враг','время','жизнь','смерть','мир','война','надежда','мечта',
           'изменение','любовь','страх','доверие','радость','боль','давать','брать','говорить','думать',
           'найти','потерять','строить','ломать','есть','чувствовать','спать','бежать','свет','тьма'],
    'pt': ['água','fogo','terra','céu','vento','pedra','rio','montanha','floresta','criança',
           'idoso','amigo','inimigo','tempo','vida','morte','paz','guerra','esperança','sonho',
           'mudança','amor','medo','confiança','alegria','dor','dar','tomar','falar','pensar',
           'encontrar','perder','construir','quebrar','comer','sentir','dormir','correr','luz','escuridão'],
    'fr': ['eau','feu','terre','ciel','vent','pierre','rivière','montagne','forêt','enfant',
           'aîné','ami','ennemi','temps','vie','mort','paix','guerre','espoir','rêve',
           'changement','amour','peur','confiance','joie','douleur','donner','prendre','parler','penser',
           'trouver','perdre','construire','casser','manger','ressentir','dormir','courir','lumière','obscurité'],
    'de': ['Wasser','Feuer','Erde','Himmel','Wind','Stein','Fluss','Berg','Wald','Kind',
           'Alter','Freund','Feind','Zeit','Leben','Tod','Frieden','Krieg','Hoffnung','Traum',
           'Veränderung','Liebe','Angst','Vertrauen','Freude','Schmerz','geben','nehmen','sprechen','denken',
           'finden','verlieren','bauen','brechen','essen','fühlen','schlafen','laufen','Licht','Dunkel'],
    'ja': ['水','火','土','空','風','石','川','山','森','子供',
           '老人','友達','敵','時間','生命','死','平和','戦争','希望','夢',
           '変化','愛','恐怖','信頼','喜び','痛み','与える','取る','話す','考える',
           '見つける','失う','建てる','壊す','食べる','感じる','寝る','走る','光','闇'],
}

print('Embedding multilingual vocabulary...')
records = []
for lang, words in LANG_VOCAB.items():
    assert len(words) == len(CONCEPTS), f'Length mismatch for {lang}'
    embs = model.encode(words, normalize_embeddings=True)
    for concept, word, emb in zip(CONCEPTS, words, embs):
        records.append({'lang': lang, 'concept': concept, 'word': word, 'emb': emb})

df = pd.DataFrame(records)

# Speaker-weighted universal centroids
centroids = {}
for concept in CONCEPTS:
    sub = df[df['concept'] == concept]
    w   = np.array([SPEAKER_WEIGHTS[l] for l in sub['lang']], dtype=float)
    w  /= w.sum()
    c   = (np.stack(sub['emb'].values) * w[:, None]).sum(axis=0)
    c  /= np.linalg.norm(c)
    centroids[concept] = c

centroid_matrix = np.stack([centroids[c] for c in CONCEPTS])
print(f'Centroid matrix: {centroid_matrix.shape}')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# PROBE 1 — Universal category emergence
# UG claim: all languages have Noun and Verb categories.
# Our test: k-means (k=5) on centroids — no labels used.
# Does clustering recover the POS-like categories spontaneously?
# ═══════════════════════════════════════════════════════════════════════════════
print('═'*60)
print('PROBE 1 — Universal category emergence')
print('═'*60)

K_CLUSTERS = 5
kmeans = KMeans(n_clusters=K_CLUSTERS, random_state=42, n_init=20)
cluster_ids = kmeans.fit_predict(centroid_matrix)

# Map cluster → best matching category via majority vote
POS_MAP = {pos: i for i, pos in enumerate(sorted(set(POS_LABELS)))}
y_true  = np.array([POS_MAP[p] for p in POS_LABELS])

# For each cluster find dominant POS
cluster_to_pos = {}
for k in range(K_CLUSTERS):
    mask      = cluster_ids == k
    members   = [CONCEPTS[i] for i in range(len(CONCEPTS)) if mask[i]]
    pos_votes = [POS_LABELS[i] for i in range(len(CONCEPTS)) if mask[i]]
    dominant  = max(set(pos_votes), key=pos_votes.count) if pos_votes else 'unknown'
    cluster_to_pos[k] = dominant
    print(f'  Cluster {k} ({dominant}): {members}')

# Majority-vote accuracy
y_pred_pos = np.array([POS_MAP[cluster_to_pos[c]] for c in cluster_ids])
acc = accuracy_score(y_true, y_pred_pos)
print(f'\nCategory recovery accuracy: {acc:.3f}')
print(f'Chance baseline: {1/len(POS_MAP):.3f}')
if acc > 2.5 * (1/len(POS_MAP)):
    print('VERDICT: EMERGENT — k-means recovers POS-like categories')
else:
    print('VERDICT: NOT_DETECTED — clusters do not align with grammatical categories')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# PROBE 2 — Argument structure (theta-criterion)
# UG: transitive verbs take agent + patient; intransitive take only agent.
# Test: are transitive/intransitive verb centroids linearly separable?
# ═══════════════════════════════════════════════════════════════════════════════
print('\n' + '═'*60)
print('PROBE 2 — Argument structure (transitivity)')
print('═'*60)

verb_idx   = [i for i, t in enumerate(TRANS) if t is not None]
verb_labels = [1 if TRANS[i] == 'transitive' else 0 for i in verb_idx]
verb_embs   = centroid_matrix[verb_idx]

print(f'Verbs: {len(verb_idx)} total  ({sum(verb_labels)} transitive, {len(verb_labels)-sum(verb_labels)} intransitive)')
print(f'Verb concepts: {[CONCEPTS[i] for i in verb_idx]}')

# Leave-one-out logistic regression
loo   = LeaveOneOut()
y_pred_trans = []
for train_idx, test_idx in loo.split(verb_embs):
    clf = LogisticRegression(max_iter=1000, random_state=42)
    clf.fit(verb_embs[train_idx], np.array(verb_labels)[train_idx])
    y_pred_trans.append(int(clf.predict(verb_embs[test_idx])[0]))

trans_acc = accuracy_score(verb_labels, y_pred_trans)
print(f'\nLOO transitivity classification accuracy: {trans_acc:.3f}')
print(f'Chance baseline: 0.500')
if trans_acc > 0.70:
    print('VERDICT: EMERGENT — transitivity is linearly encoded in the concept space')
elif trans_acc > 0.55:
    print('VERDICT: PARTIAL — some transitivity signal present, not robust')
else:
    print('VERDICT: NOT_DETECTED — centroids do not encode transitivity linearly')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# PROBE 3 — Economy / Zipf's law
# UG economy: most primitive concepts should be most central.
# Test: do pairwise centroid distances follow a power law?
# Is there a 'universal centre' that primitive concepts cluster around?
# ═══════════════════════════════════════════════════════════════════════════════
from scipy.optimize import curve_fit

print('\n' + '═'*60)
print('PROBE 3 — Economy / Zipf\'s law')
print('═'*60)

# Pairwise distances
pw_dists = pdist(centroid_matrix, metric='cosine')

# Sort and check power law
sorted_dists = np.sort(pw_dists)
ranks        = np.arange(1, len(sorted_dists)+1)

def power_law(x, a, b):
    return a * np.power(x, b)

try:
    popt, _ = curve_fit(power_law, ranks, sorted_dists, p0=[1.0, 0.1], maxfev=5000)
    fitted  = power_law(ranks, *popt)
    ss_res  = np.sum((sorted_dists - fitted)**2)
    ss_tot  = np.sum((sorted_dists - sorted_dists.mean())**2)
    r2      = 1 - ss_res/ss_tot
    print(f'Power law fit: a={popt[0]:.4f}, b={popt[1]:.4f}, R²={r2:.4f}')
except:
    r2 = 0.0
    print('Power law fit failed')

# Centrality: which concepts are most 'central' (closest to all others)?
mean_dist_from_all = squareform(pdist(centroid_matrix, metric='cosine')).mean(axis=1)
rankings = np.argsort(mean_dist_from_all)
print('\nMost central concepts (lowest mean distance to all others):')
for rank, idx in enumerate(rankings[:8]):
    print(f'  {rank+1}. {CONCEPTS[idx]:<12} (mean_dist={mean_dist_from_all[idx]:.4f}, POS={POS_LABELS[idx]})')
print('\nMost peripheral concepts:')
for rank, idx in enumerate(rankings[-5:]):
    print(f'  {CONCEPTS[idx]:<12} (mean_dist={mean_dist_from_all[idx]:.4f}, POS={POS_LABELS[idx]})')

PRIMITIVE = ['water','fire','life','death','love','fear','light','dark']
prim_idx   = [CONCEPTS.index(c) for c in PRIMITIVE if c in CONCEPTS]
prim_mean  = np.mean([mean_dist_from_all[i] for i in prim_idx])
all_mean   = np.mean(mean_dist_from_all)
print(f'\nPrimitive concepts mean centrality: {prim_mean:.4f}')
print(f'All concepts mean centrality: {all_mean:.4f}')
if prim_mean < all_mean:
    print('VERDICT: EMERGENT — primitive concepts are more central than average (Zipf-like)')
else:
    print('VERDICT: NOT_DETECTED — no centrality advantage for primitive concepts')
print(f'Power law R²: {r2:.4f} (> 0.9 = good fit)')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# PROBE 4 — Cross-linguistic RSA (deep structure universality)
# UG: deep structure is universal; surface form differs.
# Test: is D_universal better correlated with all D_lang than any D_lang with another?
# ═══════════════════════════════════════════════════════════════════════════════
print('\n' + '═'*60)
print('PROBE 4 — Cross-linguistic RSA')
print('═'*60)

def get_lang_dist_matrix(lang):
    sub  = df[df['lang'] == lang].sort_values('concept')
    embs = np.stack(sub.set_index('concept').loc[CONCEPTS, 'emb'].values)
    return squareform(pdist(embs, metric='cosine'))

D_universal = squareform(pdist(centroid_matrix, metric='cosine'))
lang_dists  = {lang: get_lang_dist_matrix(lang) for lang in LANG_VOCAB}

# Upper triangle indices (exclude diagonal)
triu = np.triu_indices(len(CONCEPTS), k=1)
d_univ_flat = D_universal[triu]

print('Spearman ρ between D_universal and D_lang:')
rho_universal = {}
for lang in LANG_VOCAB:
    d_lang_flat = lang_dists[lang][triu]
    rho, pval   = spearmanr(d_univ_flat, d_lang_flat)
    rho_universal[lang] = rho
    print(f'  D_universal vs D_{lang}: ρ={rho:.4f}  p={pval:.4f}')

print('\nSpearman ρ between individual language pairs (sample):')
lang_list = list(LANG_VOCAB.keys())
inter_lang_rhos = []
for i in range(len(lang_list)):
    for j in range(i+1, len(lang_list)):
        la, lb = lang_list[i], lang_list[j]
        d_a = lang_dists[la][triu]
        d_b = lang_dists[lb][triu]
        rho, _ = spearmanr(d_a, d_b)
        inter_lang_rhos.append(rho)

mean_inter_lang  = np.mean(inter_lang_rhos)
mean_univ_corr   = np.mean(list(rho_universal.values()))
print(f'  Mean inter-language ρ: {mean_inter_lang:.4f}')
print(f'  Mean D_universal–D_lang ρ: {mean_univ_corr:.4f}')

if mean_univ_corr > mean_inter_lang:
    print('VERDICT: EMERGENT — D_universal is a better common structure than any individual language')
else:
    print('VERDICT: NOT_DETECTED — individual languages correlate better with each other than with universal')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# PROBE 5 — Semantic directions (displacement / internal merge)
# UG: there are consistent abstract directions in conceptual space.
# Test: can we find a positive/negative axis that generalises to held-out pairs?
# ═══════════════════════════════════════════════════════════════════════════════
print('\n' + '═'*60)
print('PROBE 5 — Semantic directions')
print('═'*60)

# Define direction from TRAINING antonym pairs
TRAIN_PAIRS_POS_NEG = [
    ('love',  'fear'),
    ('life',  'death'),
    ('peace', 'war'),
    ('light', 'dark'),
    ('hope',  'pain'),
    ('joy',   'fear'),
    ('friend','enemy'),
]

# Held-out pairs — not used to define the direction
HELD_OUT_PAIRS = [
    ('trust',  'fear'),
    ('find',   'lose'),
    ('build',  'break'),
    ('give',   'take'),
    ('sleep',  'run'),
]

# Compute direction as mean(positive - negative)
direction = np.zeros(768)
for pos, neg in TRAIN_PAIRS_POS_NEG:
    if pos in centroids and neg in centroids:
        diff = centroids[pos] - centroids[neg]
        direction += diff / np.linalg.norm(diff)
direction /= np.linalg.norm(direction)

print('Positive/negative direction defined from:', [p[0]+'-'+p[1] for p in TRAIN_PAIRS_POS_NEG])

# Project all concepts onto this axis
proj = {c: float(np.dot(centroids[c], direction)) for c in CONCEPTS}
ranked_proj = sorted(proj.items(), key=lambda x: -x[1])
print('\nConcepts ranked by projection onto positive–negative axis:')
for name, score in ranked_proj:
    val_label = VALENCE[CONCEPTS.index(name)]
    print(f'  {name:<12} {score:>7.4f}  [{val_label}]')

# Check: do held-out pairs respect the direction?
print('\nHeld-out pair test (first should score higher than second):')
correct = 0
for pos, neg in HELD_OUT_PAIRS:
    if pos in proj and neg in proj:
        ok = proj[pos] > proj[neg]
        correct += int(ok)
        print(f'  {pos}({proj[pos]:+.4f}) > {neg}({proj[neg]:+.4f})  → {"✓" if ok else "✗"}')

ho_acc = correct / len(HELD_OUT_PAIRS)
print(f'\nHeld-out direction accuracy: {ho_acc:.2f}/{len(HELD_OUT_PAIRS)}')

# Also validate against valence labels
val_map = {'positive': 1, 'negative': -1, 'neutral': 0}
val_scores = np.array([val_map[v] for v in VALENCE])
proj_scores = np.array([proj[c] for c in CONCEPTS])
mask_non_neutral = val_scores != 0
rho_val, _ = spearmanr(proj_scores[mask_non_neutral], val_scores[mask_non_neutral])
print(f'Spearman ρ(projection, true valence) on non-neutral concepts: {rho_val:.4f}')

if ho_acc >= 0.8 and rho_val > 0.4:
    print('VERDICT: EMERGENT — consistent semantic direction generalises to held-out pairs')
elif ho_acc >= 0.6:
    print('VERDICT: PARTIAL — direction is present but noisy')
else:
    print('VERDICT: NOT_DETECTED — no consistent semantic direction found')

In [ ]:
# ── Summary visualisation ─────────────────────────────────────────────────────
fig = plt.figure(figsize=(18, 12))
gs  = fig.add_gridspec(2, 3, hspace=0.4, wspace=0.35)

# Plot 1: PCA of centroids coloured by POS (Probe 1)
ax1 = fig.add_subplot(gs[0, 0])
pca  = PCA(n_components=2)
C2   = pca.fit_transform(centroid_matrix)
POS_COLORS = {
    'concrete_noun':     '#1976D2',
    'abstract_noun':     '#7B1FA2',
    'emotion':           '#E91E63',
    'transitive_verb':   '#388E3C',
    'intransitive_verb': '#F57C00',
}
for pos, color in POS_COLORS.items():
    idx = [i for i, p in enumerate(POS_LABELS) if p == pos]
    ax1.scatter(C2[idx, 0], C2[idx, 1], c=color, s=60, label=pos.replace('_', ' '), alpha=0.8)
for i, concept in enumerate(CONCEPTS):
    ax1.annotate(concept, (C2[i,0]+0.002, C2[i,1]+0.002), fontsize=6)
ax1.set_title('Probe 1: PCA of centroids\n(colour = true POS, unsupervised k-means on top)', fontsize=9)
ax1.legend(fontsize=6, loc='lower left')
ax1.set_xlabel('PC1'); ax1.set_ylabel('PC2')

# Plot 2: transitivity separation (Probe 2)
ax2 = fig.add_subplot(gs[0, 1])
verb_vecs = centroid_matrix[verb_idx]
vc2 = PCA(n_components=2).fit_transform(verb_vecs)
trans_colors = ['#388E3C' if v == 1 else '#F57C00' for v in verb_labels]
ax2.scatter(vc2[:,0], vc2[:,1], c=trans_colors, s=100, alpha=0.8)
for i, idx_v in enumerate(verb_idx):
    ax2.annotate(CONCEPTS[idx_v], (vc2[i,0]+0.001, vc2[i,1]+0.001), fontsize=8)
from matplotlib.patches import Patch
ax2.legend(handles=[Patch(color='#388E3C',label='transitive'), Patch(color='#F57C00',label='intransitive')], fontsize=9)
ax2.set_title(f'Probe 2: Argument structure\nLOO accuracy={trans_acc:.2f} (chance=0.50)', fontsize=9)
ax2.set_xlabel('PC1'); ax2.set_ylabel('PC2')

# Plot 3: Zipf / centrality (Probe 3)
ax3 = fig.add_subplot(gs[0, 2])
centrality_ranks = np.argsort(mean_dist_from_all)
centrality_vals  = mean_dist_from_all[centrality_ranks]
colors_c = [POS_COLORS.get(POS_LABELS[i], 'gray') for i in centrality_ranks]
ax3.barh([CONCEPTS[i] for i in centrality_ranks], centrality_vals,
         color=colors_c, alpha=0.8)
ax3.set_xlabel('Mean cosine distance to all other concepts', fontsize=9)
ax3.set_title('Probe 3: Concept centrality\n(left = most central)', fontsize=9)
ax3.tick_params(labelsize=7)

# Plot 4: RSA heatmap (Probe 4)
ax4 = fig.add_subplot(gs[1, 0])
lang_names  = sorted(LANG_VOCAB.keys())
rho_matrix  = np.zeros((len(lang_names)+1, len(lang_names)+1))
all_names   = lang_names + ['universal']
for i, la in enumerate(all_names):
    for j, lb in enumerate(all_names):
        if i == j:
            rho_matrix[i,j] = 1.0
        else:
            d_a = (D_universal if la == 'universal' else lang_dists[la])[triu]
            d_b = (D_universal if lb == 'universal' else lang_dists[lb])[triu]
            rho_matrix[i,j], _ = spearmanr(d_a, d_b)
im = ax4.imshow(rho_matrix, cmap='YlOrRd', vmin=0, vmax=1)
ax4.set_xticks(range(len(all_names))); ax4.set_yticks(range(len(all_names)))
ax4.set_xticklabels(all_names, rotation=45, ha='right', fontsize=8)
ax4.set_yticklabels(all_names, fontsize=8)
plt.colorbar(im, ax=ax4)
ax4.set_title('Probe 4: Cross-linguistic RSA\n(universal = last row/col)', fontsize=9)

# Plot 5: semantic direction projection (Probe 5)
ax5 = fig.add_subplot(gs[1, 1:])
concepts_sorted = [name for name, _ in ranked_proj]
scores_sorted   = [score for _, score in ranked_proj]
val_colors_sorted = [
    '#388E3C' if VALENCE[CONCEPTS.index(c)] == 'positive'
    else '#E91E63' if VALENCE[CONCEPTS.index(c)] == 'negative'
    else '#9E9E9E'
    for c in concepts_sorted
]
bars = ax5.barh(concepts_sorted, scores_sorted, color=val_colors_sorted, alpha=0.8)
ax5.axvline(0, color='black', linewidth=1)
ax5.set_xlabel('Projection onto positive/negative direction', fontsize=10)
ax5.set_title(f'Probe 5: Semantic direction generalisation\n(held-out acc={ho_acc:.2f}, valence ρ={rho_val:.3f})', fontsize=9)
from matplotlib.patches import Patch
ax5.legend(handles=[
    Patch(color='#388E3C', label='positive valence'),
    Patch(color='#E91E63', label='negative valence'),
    Patch(color='#9E9E9E', label='neutral'),
], fontsize=9, loc='lower right')
ax5.tick_params(labelsize=8)

fig.suptitle('Experiment 9 — Universal Grammar Structural Probes\nDo UG-like properties emerge from the multilingual concept space?',
             fontsize=13, fontweight='bold')
plt.savefig('exp9_ug_probes.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved.')

In [ ]:
# ── Final summary ─────────────────────────────────────────────────────────────
print('\n' + '═'*60)
print('EXPERIMENT 9 — SUMMARY')
print('═'*60)
print(f'  Probe 1 (Category emergence)  : acc={acc:.3f} (chance={1/len(POS_MAP):.3f})')
print(f'  Probe 2 (Argument structure)  : LOO acc={trans_acc:.3f} (chance=0.500)')
print(f'  Probe 3 (Economy/Zipf)        : R²={r2:.4f}, primitive centrality={prim_mean:.4f} vs all={all_mean:.4f}')
print(f'  Probe 4 (Cross-linguistic RSA): mean univ–lang ρ={mean_univ_corr:.4f} vs inter-lang ρ={mean_inter_lang:.4f}')
print(f'  Probe 5 (Semantic directions) : held-out acc={ho_acc:.2f}, valence ρ={rho_val:.4f}')
print()
print('Implication:')
print('  If probes 1–4 are EMERGENT: UG structure arises from communication pressure,')
print('  not biological imposition. A data-driven universal language gets UG for free.')
print('  If NOT_DETECTED: UG structure must be explicitly designed in — see Exp 10.')